# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from adaptivelasso.skcompat import MAPAdaptiveLasso
from adaptivelasso.utils import generate_data, report_model_results
import numpy as np

## Compare MAPAdaptiveLasso with LassoCV

In [2]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

ES stop!
ES stop!
Estimated coefficients: [ 1.34543251e+00  1.34544432e+00  1.22827788e+00  1.36146619e+00
  1.36443837e+00  3.38394358e-02  1.65747310e-01  1.51566283e-01
  6.55682827e-02 -2.99462744e-02 -9.66660774e-02  1.14003002e-01
  7.85721607e-02  1.30524943e-01 -8.36840997e-02 -5.43380968e-02
  0.00000000e+00 -0.00000000e+00  1.37756660e-01 -3.16348111e-04]
R²: 0.5737
RMSE: 0.3241
Estimated coefficients: [ 0.16488544  0.1550696   0.14862804  0.16592588  0.16575047 -0.
  0.01683911  0.01516307  0.0040984  -0.0040288  -0.          0.01580763
 -0.          0.00102194  0.         -0.00028995  0.00109467  0.00600582
  0.01435396  0.        ]
R²: 0.4902
RMSE: 0.3544


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb', 'nb']
results = []
max_iters = 3000  # Set a maximum number of iterations
tol = 1e-4  # Set a tolerance for convergence
cv = 5

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=True, tol=tol, cv=cv))
    ])
    pipe.fit(X_train, y_train)
    y_pred_adapt = pipe.predict(X_test)
    r2_adapt = r2_score(y_test, y_pred_adapt)
    rmse_adapt = np.sqrt(mean_squared_error(y_test, y_pred_adapt))
    coefs_adapt = pipe.named_steps['jax_model'].get_coefficients()
    coefs_adapt_rounded = np.round(coefs_adapt, 3).tolist()

    # Always fit LassoCV, regardless of likelihood
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=cv, random_state=0, max_iter=max_iters, tol=tol))
    ])
    pipe_lasso.fit(X_train, y_train)
    y_pred_lasso = pipe_lasso.predict(X_test)
    if lik == 'bernoulli':
        y_pred_lasso_bin = (y_pred_lasso > 0.5).astype(int)
        r2_lasso = r2_score(y_test, y_pred_lasso_bin)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso_bin))
    else:
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    coefs_lasso = pipe_lasso.named_steps['lasso'].coef_
    coefs_lasso_rounded = np.round(coefs_lasso, 3).tolist()

    better_r2 = 'AdaptiveLasso' if r2_adapt > r2_lasso else 'LassoCV'
    better_rmse = 'AdaptiveLasso' if rmse_adapt < rmse_lasso else 'LassoCV'

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': r2_adapt,
        'RMSE_AdaptiveLasso': rmse_adapt,
        'Coefs_AdaptiveLasso': coefs_adapt_rounded[:5],  # Show first 5 for brevity
        'R2_LassoCV': r2_lasso,
        'RMSE_LassoCV': rmse_lasso,
        'Coefs_LassoCV': coefs_lasso_rounded[:5],        # Show first 5 for brevity
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
ES stop!
ES stop!
ES stop!
Testing likelihood: cauchy
ES stop!
ES stop!
Testing likelihood: bernoulli
ES stop!
ES stop!
Testing likelihood: poisson
ES stop!
ES stop!
ES stop!
Testing likelihood: zinb
ES stop!
ES stop!
Testing likelihood: nb
ES stop!
ES stop!
ES stop!
  likelihood  R2_AdaptiveLasso  RMSE_AdaptiveLasso  \
0     normal          0.728002            0.951864   
1     cauchy          0.020220            7.062369   
2  bernoulli          0.203724            0.445368   
3    poisson          0.965137            2.497156   
4       zinb          0.400642           11.540609   
5         nb          0.477290           14.108393   

                    Coefs_AdaptiveLasso  R2_LassoCV  RMSE_LassoCV  \
0  [0.993, 1.058, 1.055, 0.013, -0.025]    0.736476      0.936920   
1   [1.05, 1.039, 0.872, 0.092, -0.052]    0.000429      7.133340   
2       [0.9, 0.828, 0.852, 0.064, 0.0]   -0.284625      0.565685   
3  [0.972, 1.016, 0.988, -0.014, 0.002]  